In [1]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold
import pandas as pd
import numpy as np
import joblib
import os

In [2]:
# =========================== Eliminarlo e Importarlo desde el .py =========================== #

# Métricas de evaluación utilizadas en el paper original

def paper_r2(y_true, y_pred):
    """Paper's metric: Pearson correlation"""
    corr = np.corrcoef(y_true, y_pred)[0, 1]
    return corr 

def rmse(y_true, y_pred):
    return np.sqrt(np.mean((y_true - y_pred)**2))

In [3]:
# Definition of attributes and targets

# =========================== 21 atributos seleccionados =========================== #

selected_21_attributes = [
    'np_without_modification', 'surface_modification', 'size_tem', 'zeta_potential', 
    'incubation_protein_source', 'incubation_plasma_concentration', 'incubation_np_concentration',
    'centrifugation_speed', 'centrifugation_time', 'centrifugation_temperature', 'np_type',
    'np_shape', 'dispersion_medium', 'dispersion_medium_ph', 'size_dls', 'pdi', 'incubation_culture',
    'incubation_time', 'incubation_temperature', 'centrifugation_repetitions', 'modification_type'
]


categorical_21_attributes = [
    'np_without_modification', 'surface_modification', 'incubation_protein_source',
    'np_type', 'np_shape', 'dispersion_medium', 'incubation_culture', 'modification_type'
]

# =========================== 10 atributos seleccionados =========================== #

selected_10_attributes = [
    'np_without_modification', 'surface_modification', 
    'incubation_protein_source', 'size_tem', 'zeta_potential', 'incubation_plasma_concentration', 
    'incubation_np_concentration','centrifugation_speed', 
    'centrifugation_time', 'centrifugation_temperature'    
]

categorical_10_attributes = [
    'np_without_modification', 'surface_modification', 'incubation_protein_source',
]

# =========================== 17 atributos seleccionados (Jair) =========================== #

selected_17_attributes = [
    'np_without_modification', 'surface_modification', 'zeta_potential', 
    'incubation_protein_source', 'incubation_plasma_concentration', 'incubation_np_concentration',
    'np_type', 'np_shape', 'dispersion_medium', 'dispersion_medium_ph', 'size_dls', 'pdi', 
    'incubation_culture', 'incubation_time', 'incubation_temperature', 'modification_type'
]

categorical_17_attributes = [
    'np_without_modification', 'surface_modification', 'incubation_protein_source',
    'np_type', 'np_shape', 'dispersion_medium', 'incubation_culture', 'modification_type'
]

# =========================== 16 atributos seleccionados (sin np_type para el entrenamiento) =========================== #

selected_16_attributes = [
    'np_without_modification', 'surface_modification', 'zeta_potential', 
    'incubation_protein_source', 'incubation_plasma_concentration', 'incubation_np_concentration', 
    'np_shape', 'dispersion_medium', 'dispersion_medium_ph', 'size_dls', 'pdi', 
    'incubation_culture', 'incubation_time', 'incubation_temperature', 'modification_type'
]

categorical_16_attributes = [
    'np_without_modification', 'surface_modification', 'incubation_protein_source',
    'np_shape', 'dispersion_medium', 'incubation_culture', 'modification_type'
]

# =========================== 178 Proteinas =========================== #

protein_targets = [
    "P01871", "P01024", "P02647", "P02649", "P02768", "P04004", "P01834", 
    "P10909", "P02652", "P00734", "P01009", "P01042", "P01857", "P01859", 
    "P01023", "P01619", "P04196", "P02656", "P04114", "P02787", "P02766", 
    "P06396", "P06727", "P0C0L5", "P02671", "P08603", "P02765", "Q14624", 
    "P0DOY2", "P01008", "P68871", "P01764", "P04003", "P0C0L4", "P01876", 
    "P02749", "P02675", "P01860", "P01011", "P00747", "P02774", "P00751", 
    "P19823", "P08697", "P19827", "P69905", "P02760", "P02748", "P02679", 
    "P02790", "B9A064", "P07996", "P05155", "P01766", "P12259", "P00738",
    "P02751", "P02654", "P04406", "P00739", "P02655", "P00736", "P07225", 
    "P05154", "P09871", "P60709", "P03952", "P02747", "P05546", "P02746",
    "P27169", "P01019", "P35542", "P04217", "Q14520", "P02743", "P02763", 
    "P49908", "O43866", "P18428", "P55056", "P04264", "P00748", "P01591", 
    "P01861", "Q92954", "P02776", "P0DJI8", "P01031", "P05156", "P13671", 
    "Q03591", "P06312", "P00740", "P01615", "P00742", "P04070", "O14791", 
    "P05090", "P02745", "P00488", "Q13103", "P01877", "P01700", "P10643", 
    "Q9UK55", "P03951", "Q06033", "Q96IY4", "P13645", "P04433", "P07358",
    "P27918", "P05452", "P20851", "P07357", "P07360", "P01599", "O95445", 
    "Q9BXR6", "P02775", "P02741", "P35579", "P15169", "Q13790", "P35858", 
    "P49747", "P36955", "P19652", "P07737", "P22891", "P35908", "P80748", 
    "P08514", "P01593", "Q04756", "P06733", "P23528", "P63104", "P18065", 
    "P08519", "Q86UX7", "P02753", "Q5TB80", "P22352", "P00746", "P35443", 
    "P62937", "P10720", "P48740", "P25311", "P43652", "P35527", "Q9Y490", 
    "P05106", "P17936", "P18206", "P02788", "P06702", "P01701", "Q6Q788", 
    "Q96PD5", "Q9UGM5", "O00391", "P11142", "P14618", "P23142", "P68366",
    "P12814", "P61224", "Q13201", "P67936", "P0DOY3", "P08709", "P01034", 
    "P81605", "P02533", "P11226"]

In [4]:
def predict_csv(csv_filename, selected_attributes, categorical_attributes, protein_targets, directory='./modelos_guardados/', show=False, output_filename=''):
    """
    Carga nuevos datos, aplica One-Hot Encoding, alinea las columnas basándose en 
    el archivo guardado durante el entrenamiento, y genera predicciones.
    """
    if not all([selected_attributes, categorical_attributes, protein_targets]):
        print("Error: Faltan especificar atributos, variables categóricas o targets.")
        return

    # Cargamos el dataset o el archivo .csv
    dj = pd.read_csv(f"{csv_filename}.csv")
    # Creamos una copia para no ensuciar el DataFrame original
    predictions_df = pd.DataFrame() 
    # Seleccionamos los atributos de interes del dataset
    X_test = dj[selected_attributes]
    # Hacemos la codificación One-hot a los atributos categóricos en caso que haya
    X_test = pd.get_dummies(X_test, columns=categorical_attributes)

    # Cargamos la estructura de columnas utilizadas en el entrenamiento
    columns_filepath = os.path.join(directory, 'training_columns.joblib')
    try:
        training_columns = joblib.load(columns_filepath)
    except FileNotFoundError:
        print(f"Error: No se encontró el archivo de columnas en {columns_filepath}.")
        print("Asegúrate de ejecutar fit_csv primero para el directorio especificado.")
        return

    # 3. Alinear las columnas del dataset de prueba
    X_test = X_test.reindex(columns=training_columns, fill_value=0)
    
    # 4. Carga de Modelos y Predicción
    for target in protein_targets:
        filename = f'random_forest_regressor_model_{target}.joblib'
        filepath = os.path.join(directory, filename)
        
        try:
            loaded_model = joblib.load(filepath)
        except FileNotFoundError:
            print(f"Advertencia: Modelo para {target} no encontrado en {filepath}. Se omite.")
            continue

        loaded_prediction = loaded_model.predict(X_test)
        
        # Añadimos la predicción a nuestro DataFrame temporal
        predictions_df[f"{target}_prediccion"] = loaded_prediction

        if show:
            print(f"============= Proteína: {target} =============")
            print(f"Predicción: {loaded_prediction[:5]}...") # Muestra solo los primeros 5 para legibilidad

    # 5. Concatenar y Guardar
    # Unimos el DataFrame original con todas las predicciones a la vez para mejorar el rendimiento
    dj_final = pd.concat([dj, predictions_df], axis=1)
    
    # output_filename = f"{csv_filename}_con_predicciones.csv"
    dj_final.to_csv(output_filename, index=False)
    
    print("\n--- PROCESO TERMINADO ---")
    print(f"Archivo final guardado como: {output_filename}")
    print(f"Proteínas predichas: {len(predictions_df.columns)}")

In [ ]:
# Predicción con el modelo que fue entrenado con las NP de tipo Liposomas sin el atributo 'np_type'

# Definimos el nombre del archivo a leer 
csv_filename_predict="datos_np_unq_externo"
output_filename = f"{csv_filename_predict}-prediction_lip-np_type.csv"  # Asignar un nombre correspondiente a los modelos utilizados

# Definimos la ruta de donde se encuentra el las NPs a predecir
input_filepath = os.path.join('csv_to_predict', csv_filename_predict)

# Definimos la ruta de donde se leeran los modelos
model_filename='liposome_models-np_type'
model_directory = os.path.join("models", model_filename)
os.makedirs(model_directory, exist_ok=True)

output_directory = "csv_with_predictions"
os.makedirs(output_directory, exist_ok=True)
output_filepath = os.path.join(output_directory, output_filename)

predict_csv(
    csv_filename=input_filepath,
    selected_attributes=selected_16_attributes, 
    categorical_attributes=categorical_16_attributes, 
    protein_targets=protein_targets, 
    directory=model_directory,
    output_filename=output_filepath
)

In [ ]:
# Predicción con el modelo que fue entrenado con todo el dataset.

csv_filename_predict="datos_np_unq_externo"
directory='./modelos_guardados'
output_filename = f"{csv_filename_predict}_prediction.csv"
predict_csv(csv_filename_predict ,selected_21_attributes, categorical_21_attributes, protein_targets, directory, output_filename=output_filename)

In [ ]:
# Predicción con los modelos optimizados que fueron entrenados con las NPs de tipo liposomas sin el atributo 'np_type'

csv_filename_predict="datos_np_unq_externo"
directory='./liposome_models_opt-np_type'
output_filename = f"{csv_filename_predict}_prediction_opt_lip-np_type.csv"

predict_csv(
    csv_filename=csv_filename_predict,
    selected_attributes=selected_16_attributes, 
    categorical_attributes=categorical_16_attributes, 
    protein_targets=protein_targets, 
    directory=directory,
    output_filename=output_filename
)


In [5]:
# Predicción con el modelo que fue entrenado con las NP de tipo Liposomas sin el atributo 'np_type'

# Definimos el nombre del archivo a leer 
csv_filename_predict="datos_np_unq_externo"
output_filename = f"{csv_filename_predict}-prediction_opt_lip-np_type.csv"  # Asignar un nombre correspondiente a los modelos utilizados

# Definimos la ruta de donde se encuentra el las NPs a predecir
input_filepath = os.path.join('csv_to_predict', csv_filename_predict)

# Definimos la ruta de donde se leeran los modelos
model_filename='liposome_models_opt-np_type'
model_directory = os.path.join("models", model_filename)
os.makedirs(model_directory, exist_ok=True)

output_directory = "csv_with_predictions"
os.makedirs(output_directory, exist_ok=True)
output_filepath = os.path.join(output_directory, output_filename)

predict_csv(
    csv_filename=input_filepath,
    selected_attributes=selected_16_attributes, 
    categorical_attributes=categorical_16_attributes, 
    protein_targets=protein_targets, 
    directory=model_directory,
    output_filename=output_filepath
)

C:\Users\emanu\miniconda3\envs\proteina-corona\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator DecisionTreeRegressor from version 1.7.2 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
C:\Users\emanu\miniconda3\envs\proteina-corona\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator RandomForestRegressor from version 1.7.2 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
C:\Users\emanu\miniconda3\envs\proteina-corona\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator Decisio


--- PROCESO TERMINADO ---
Archivo final guardado como: csv_with_predictions\datos_np_unq_externo-prediction_opt_lip-np_type.csv
Proteínas predichas: 178


C:\Users\emanu\AppData\Local\Temp\ipykernel_6408\872833595.py:45: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  predictions_df[f"{target}_prediccion"] = loaded_prediction
